# PIDA-RL Diabetes — 10. Cumplimiento de la Definición del PIDA (V2)

Este notebook evalúa los criterios de éxito de la sección 5 de la Definición del PIDA sobre el entorno V2. Usa la capacidad estricta, los costos 0/1/3/5 y las acciones 0–3, donde 3 es teleorientación.

| Criterio | Cómo se mide aquí |
|---|---|
| Validez técnica | `pytest` del repositorio, que incluye `check_env` de Gymnasium y los invariantes |
| Factibilidad operativa | Violaciones de capacidad en todos los episodios evaluados |
| Comparación | Seis políticas con las mismas semillas y escenarios |
| Rendimiento | Recompensa media con IC95 en **100 episodios**, frente a la mejor referencia |
| Prioridad clínica simulada | Cobertura de riesgo alto del mejor agente frente a la política por reglas |
| Robustez | 7 escenarios: capacidad ×0.5 y ×1.5, costos bajos y altos, riesgo basal ×0.75 y ×1.25, efecto del contacto ×0.5 |
| Equidad operativa | Brecha de cobertura entre subgrupos de sexo, edad y entidad federativa (meta ≤ 10 pp) |

Desde V2.16 el entorno baraja el orden de atención cada mes y los agentes descuentan por mes (γ^t, t = mes). Ambos cambios se fijaron antes de ver los resultados; la sección 12 conserva la comparación con el descuento por decisión.

Requisitos previos:
- Los notebooks 03 y 05–09 corregidos deben haberse ejecutado.
- Los modelos se leen de `RESULTS_V2_DIR`.

Alcance: los resultados comparan políticas dentro de un simulador. No demuestran efectividad clínica ni causalidad. Las categorías bajo, medio y alto son reglas de simulación, no una escala clínica validada.

## 1. Configuración del entorno de trabajo

In [ ]:
# === Bootstrap portable: PROJECT_ROOT (corrección v2.12) ===
# PROJECT_ROOT = raíz persistente de datos, modelos y resultados.
# REPO_ROOT    = copia local del código (repositorio GitHub).
# Prioridad: variables de entorno PIDA_PROJECT_ROOT / PIDA_REPO_ROOT,
# después Google Colab + Drive y, por último, ejecución local.
from pathlib import Path
from datetime import datetime, timezone
import os
import sys
import json
import importlib
import importlib.util
import subprocess

PROJECT_NAME = 'pida-rl-datos-publicos'

REPO_URL = 'https://github.com/JssSalas/pida-rl-datos-publicos.git'
BRANCH = os.environ.get('PIDA_BRANCH', 'main')
PROJECT_FOLDER_NAME = 'PIDA-RL-Diabetes'
IN_COLAB = 'google.colab' in sys.modules

DRIVE_ROOT = None
if os.environ.get('PIDA_PROJECT_ROOT'):
    PROJECT_ROOT = Path(os.environ['PIDA_PROJECT_ROOT']).expanduser().resolve()
elif IN_COLAB:
    DRIVE_ROOT = next(
        (p for p in [Path('/content/drive'), Path('/content/gdrive')] if (p / 'MyDrive').is_dir()),
        None,
    )
    if DRIVE_ROOT is None:
        from google.colab import drive
        mount_point = Path('/content/drive')
        if mount_point.exists() and any(mount_point.iterdir()):
            mount_point = Path('/content/gdrive')
        drive.mount(str(mount_point), force_remount=False)
        DRIVE_ROOT = mount_point
    PROJECT_ROOT = DRIVE_ROOT / 'MyDrive' / PROJECT_FOLDER_NAME
else:
    PROJECT_ROOT = (Path.cwd() / PROJECT_FOLDER_NAME).resolve()
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)


def _buscar_repo(inicio):
    for candidato in [inicio, *inicio.parents]:
        if (candidato / 'src' / 'environment' / 'diabetes_followup_env.py').is_file():
            return candidato
    return None


if os.environ.get('PIDA_REPO_ROOT'):
    REPO_ROOT = Path(os.environ['PIDA_REPO_ROOT']).expanduser().resolve()
else:
    REPO_ROOT = _buscar_repo(Path.cwd().resolve()) or (
        Path('/content/pida-rl-datos-publicos') if IN_COLAB
        else Path.cwd().resolve() / 'pida-rl-datos-publicos'
    )
if not REPO_ROOT.exists():
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_ROOT)])
elif not (REPO_ROOT / 'src').is_dir():
    raise RuntimeError(f'REPO_ROOT no contiene la carpeta src: {REPO_ROOT}')
elif IN_COLAB and (REPO_ROOT / '.git').is_dir():
    # En Colab se alinea la copia temporal con la rama de trabajo (sin push ni commit).
    for args in (['fetch', 'origin', BRANCH], ['checkout', BRANCH], ['pull', '--ff-only', 'origin', BRANCH]):
        subprocess.run(['git', '-C', str(REPO_ROOT), *args], check=False)

SRC_ROOT = REPO_ROOT / 'src'
for ruta in (str(REPO_ROOT), str(SRC_ROOT)):
    if ruta not in sys.path:
        sys.path.insert(0, ruta)
os.chdir(REPO_ROOT)

# Estructura persistente común (antes: rutas fijas /content/drive/MyDrive/PIDA-RL-Diabetes).
PERSISTENT_ROOT = PROJECT_ROOT  # alias de compatibilidad con los notebooks originales
MYDRIVE_ROOT = PROJECT_ROOT.parent
DATA_ROOT = PROJECT_ROOT / 'data'
RAW_DATA_DIR = DATA_ROOT / 'raw'
INTERIM_DATA_DIR = DATA_ROOT / 'interim'
PROCESSED_DATA_DIR = DATA_ROOT / 'processed'
METADATA_DIR = DATA_ROOT / 'metadata'
SPECIFICATIONS_DIR = DATA_ROOT / 'specifications'
MODELS_DIR = PROJECT_ROOT / 'models'
RESULTS_DIR = PROJECT_ROOT / 'results'
COHORT_PATH = PROCESSED_DATA_DIR / 'cohorte_diabetes.csv'

# Modo prueba: cohorte pequeña y pocos episodios (PIDA_MODO_PRUEBA=1).
MODO_PRUEBA = os.environ.get('PIDA_MODO_PRUEBA', '0') == '1'
N_COHORTE_PRUEBA = int(os.environ.get('PIDA_N_COHORTE', '120'))
SEMILLA_MUESTRA_PRUEBA = 2026
RESULTS_V2_DIR = RESULTS_DIR / ('capacidad_v2_prueba' if MODO_PRUEBA else 'capacidad_v2')


def muestra_cohorte_prueba(cohorte, n=None, semilla=SEMILLA_MUESTRA_PRUEBA):
    """Devuelve (cohorte, factor). En modo prueba toma una submuestra
    estratificada por categoria_riesgo (bajo/medio/alto); no crea columnas."""
    n = N_COHORTE_PRUEBA if n is None else int(n)
    if not MODO_PRUEBA or len(cohorte) <= n:
        return cohorte, 1.0
    fraccion = n / len(cohorte)
    muestra = (
        cohorte.groupby('categoria_riesgo', group_keys=False)
        .sample(frac=fraccion, random_state=semilla)
        .sort_values('FOLIO_INT')
        .reset_index(drop=True)
    )
    return muestra, len(muestra) / len(cohorte)


def escalar_capacidad(capacidad, factor):
    """Mantiene la presión de capacidad (recursos/persona) al reducir la cohorte."""
    return capacidad if factor >= 1.0 else max(1, int(round(capacidad * factor)))

print('Entorno:', 'Google Colab' if IN_COLAB else 'local')
print('PROJECT_ROOT (datos/resultados):', PROJECT_ROOT)
print('REPO_ROOT (código):', REPO_ROOT)
print('Modo prueba:', MODO_PRUEBA)

for _mod, _pip in [('gymnasium', 'gymnasium'), ('pandas', 'pandas'), ('matplotlib', 'matplotlib')]:
    if importlib.util.find_spec(_mod) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', _pip])
try:
    COMMIT = subprocess.check_output(['git', '-C', str(REPO_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
except Exception:
    COMMIT = 'no disponible'
print('Commit evaluado:', COMMIT)

## 2. Parámetros del protocolo

In [ ]:
from src.config import DEFAULT_ACTION_COSTS, DEFAULT_MONTHLY_CAPACITY
from src.evaluation import pida

CAPACIDAD_MENSUAL = DEFAULT_MONTHLY_CAPACITY
COSTOS_ACCION = dict(DEFAULT_ACTION_COSTS)
HORIZONTE_MESES = 12
GAMMA = 0.95
SEMILLAS_BASE = pida.SEMILLAS_BASE_PIDA            # 100 episodios
SEMILLAS_ROBUSTEZ = pida.SEMILLAS_ROBUSTEZ_PIDA    # 20 episodios por escenario
REFERENCIAS = ['Aleatoria', 'Reglas simples', 'Reglas escalonadas']
AGENTES = ['Q-learning', 'DQN', 'PPO']
if MODO_PRUEBA:
    SEMILLAS_BASE = tuple(range(5000, 5010))
    SEMILLAS_ROBUSTEZ = tuple(range(5000, 5004))
OUTPUT_DIR = RESULTS_V2_DIR / 'cumplimiento_pida'
CACHE_DIR = OUTPUT_DIR / 'cache'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Episodios base:', len(SEMILLAS_BASE), '| episodios por escenario de robustez:', len(SEMILLAS_ROBUSTEZ))
print('Salida:', OUTPUT_DIR)

## 3. Cohorte y modelos

In [ ]:
import hashlib
import pandas as pd

cohorte = pd.read_csv(COHORT_PATH, dtype={'FOLIO_INT': 'string'})
assert set(cohorte['categoria_riesgo'].unique()) <= {'bajo', 'medio', 'alto'}
N_COHORTE_COMPLETA = len(cohorte)
cohorte, FACTOR_COHORTE = muestra_cohorte_prueba(cohorte)
CAPACIDAD_MENSUAL = escalar_capacidad(CAPACIDAD_MENSUAL, FACTOR_COHORTE)
print(f'Cohorte: {len(cohorte)} de {N_COHORTE_COMPLETA} perfiles | capacidad mensual: {CAPACIDAD_MENSUAL}')
print(cohorte['categoria_riesgo'].value_counts().to_dict())

from src.policies.baselines import POLITICAS_BASE
from src.policies.q_learning import QLearningAgent, crear_politica_q_learning
from src.policies.dqn import DQNAgent, crear_politica_dqn
from src.policies.ppo import PPOAgent, crear_politica_ppo

MODELOS = {
    'Q-learning': (RESULTS_V2_DIR / 'q_learning' / 'q_learning_v2.npz', QLearningAgent, crear_politica_q_learning),
    'DQN': (RESULTS_V2_DIR / 'dqn' / 'dqn_v2.npz', DQNAgent, crear_politica_dqn),
    'PPO': (RESULTS_V2_DIR / 'ppo' / 'ppo_v2.npz', PPOAgent, crear_politica_ppo),
}
faltantes = [str(r) for r, _, _ in MODELOS.values() if not r.is_file()]
if faltantes:
    raise FileNotFoundError('Ejecuta los notebooks 06–08 corregidos. Faltan: ' + ', '.join(faltantes))

politicas = dict(POLITICAS_BASE)
HUELLAS_POLITICA = {n: 'regla' for n in politicas}
for nombre, (ruta, clase, fabrica) in MODELOS.items():
    politicas[nombre] = fabrica(clase.cargar(ruta, seed=20_000))
    HUELLAS_POLITICA[nombre] = hashlib.sha256(ruta.read_bytes()).hexdigest()[:16]
assert list(politicas) == REFERENCIAS + AGENTES
pd.DataFrame([{'politica': n, 'huella_modelo': h} for n, h in HUELLAS_POLITICA.items()])

## 4. Evaluación en escenarios comunes

Cada política se evalúa con las mismas semillas en cada escenario. Los resultados se guardan en `cache/` junto con una huella de la configuración: cohorte, escenario, semillas y archivo del modelo.

Si la configuración no cambia, una nueva ejecución reutiliza esos resultados. Cualquier cambio produce una huella distinta y obliga a recalcular.

In [ ]:
import time

ESCENARIOS = pida.escenarios_pida(CAPACIDAD_MENSUAL)
episodios, subgrupos, bitacora = [], [], []
for escenario in ESCENARIOS:
    semillas = SEMILLAS_BASE if escenario.nombre == 'base' else SEMILLAS_ROBUSTEZ
    for nombre, politica in politicas.items():
        t0 = time.time()
        ep, sub, en_cache = pida.evaluar_con_cache(
            escenario, cohorte, nombre, politica, semillas, CACHE_DIR, HUELLAS_POLITICA[nombre], HORIZONTE_MESES, GAMMA)
        episodios.append(ep)
        subgrupos.append(sub)
        bitacora.append({'escenario': escenario.nombre, 'politica': nombre, 'episodios': len(ep),
                         'origen': 'caché (misma huella)' if en_cache else 'calculado', 'segundos': round(time.time() - t0, 1)})
episodios = pd.concat(episodios, ignore_index=True)
subgrupos = pd.concat(subgrupos, ignore_index=True)
pd.DataFrame(bitacora).pivot_table(index='escenario', columns='politica', values='episodios', sort=False)

## 5. Invariantes de la evaluación

In [ ]:
assert episodios['violaciones_capacidad'].sum() == 0, 'Se detectaron violaciones de capacidad.'
assert set(subgrupos.loc[subgrupos['dimension'] == 'categoria_riesgo', 'subgrupo']) <= {'bajo', 'medio', 'alto'}
assert (episodios['pasos'] == len(cohorte) * HORIZONTE_MESES).all()
assert episodios.groupby(['escenario', 'algoritmo'])['semilla'].nunique().groupby('escenario').nunique().eq(1).all()
print('Episodios evaluados:', len(episodios), '| decisiones simuladas:', int(episodios['pasos'].sum()))
print('Violaciones de capacidad: 0 | categorías de riesgo: bajo, medio y alto')

## 6. Rendimiento con intervalo de confianza (escenario base)

In [ ]:
resumen = pida.resumen_con_ic(episodios)
base = resumen[resumen['escenario'] == 'base']
rendimiento = base.pivot_table(index='algoritmo', columns='metrica', values='media').reindex(REFERENCIAS + AGENTES)
ic = base[base['metrica'] == 'recompensa_acumulada'].set_index('algoritmo')[
    ['n_episodios', 'media', 'desviacion', 'ic95_inf', 'ic95_sup', 'ic95_boot_inf', 'ic95_boot_sup']]
display(ic.reindex(REFERENCIAS + AGENTES).round(2))
display(rendimiento.round(4))

In [ ]:
import matplotlib.pyplot as plt

orden = ic.reindex(REFERENCIAS + AGENTES)
fig, ax = plt.subplots(figsize=(8, 3.8))
colores = ['#8c918d'] * len(REFERENCIAS) + ['#176d68'] * len(AGENTES)
ax.barh(orden.index, orden['media'], xerr=[orden['media'] - orden['ic95_inf'], orden['ic95_sup'] - orden['media']],
        color=colores, capsize=4)
ax.set_xlabel('Recompensa acumulada media por episodio (IC95)')
ax.set_title(f'Escenario base, {int(orden["n_episodios"].min())} episodios por política')
ax.invert_yaxis()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'rendimiento_ic95.png', dpi=150)
plt.show()

## 7. Robustez: capacidad, costos y transición

In [ ]:
robustez = pida.comparar_robustez(episodios, AGENTES, REFERENCIAS)
display(robustez.round(3))
medias_esc = resumen[resumen['metrica'] == 'recompensa_acumulada'].pivot_table(
    index='escenario', columns='algoritmo', values='media', sort=False)[REFERENCIAS + AGENTES]
display(medias_esc.round(1))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.8))
r = robustez.set_index('escenario')
ax.barh(r.index, 100 * r['mejora_relativa'], color=['#176d68' if v else '#a54242' for v in r['rl_mejora']])
ax.axvline(0, color='#28251d', lw=0.8)
ax.axvline(100 * pida.META_MEJORA_ASPIRACIONAL, color='#d18f2f', ls='--', lw=0.8, label='Meta aspiracional (10 %)')
ax.set_xlabel('Mejor agente RL frente a la mejor referencia (% de recompensa)')
ax.legend(loc='lower right')
ax.invert_yaxis()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'robustez_escenarios.png', dpi=150)
plt.show()

## 8. Equidad operativa por sexo, grupo de edad y entidad federativa

In [ ]:
brechas = pida.brechas_subgrupos(subgrupos)
brechas_base = brechas[brechas['escenario'] == 'base']
display(brechas_base.round(3))
cob_sub = (subgrupos[subgrupos['escenario'] == 'base']
           .groupby(['algoritmo', 'dimension', 'subgrupo'])[['decisiones', 'contactos']].sum())
cob_sub['cobertura_pct'] = 100 * cob_sub['contactos'] / cob_sub['decisiones']
display(cob_sub['cobertura_pct'].unstack(['dimension', 'subgrupo']).reindex(REFERENCIAS + AGENTES).round(2))

## 9. Validez técnica: pruebas automáticas

In [ ]:
prueba = subprocess.run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider'], cwd=REPO_ROOT,
                        capture_output=True, text=True)
ultima = (prueba.stdout.strip().splitlines() or [''])[-1]
print(ultima)
import re
_m = re.search(r'(\d+) passed', ultima)
_f = re.search(r'(\d+) failed', ultima)
PRUEBAS = f"{_m.group(1) if _m else 0} aprobadas, {_f.group(1) if _f else 0} fallidas" + (' (con omisiones)' if 'skipped' in ultima else '')
PRUEBAS_OK = prueba.returncode == 0

## 10. Tabla de criterios de éxito

In [ ]:
criterios, clave = pida.tabla_criterios(base, episodios[episodios['escenario'] == 'base'], robustez, brechas_base,
                                        pruebas_aprobadas=PRUEBAS, agentes=tuple(AGENTES), referencias=tuple(REFERENCIAS))
criterios.loc[criterios['dimension'] == 'Validez técnica', 'estado'] = 'Cumple' if PRUEBAS_OK else 'No cumple'
pd.set_option('display.max_colwidth', 140)
display(criterios)
print('Mejor referencia:', clave['mejor_referencia'], '| mejor agente:', clave['mejor_agente'],
      f"| mejora relativa: {100 * clave['mejora_relativa']:+.1f} %")

## 11. Verificación del diccionario de datos (sección 6 del PIDA)

La sección 6 marca varias variables como «por verificar». Esta celda contrasta esas variables con la cohorte y con los archivos ENSANUT 2022 de `data/raw/`.

No crea ni añade columnas a la cohorte: solo informa disponibilidad y cobertura.

In [ ]:
import numpy as np

cohorte_completa = pd.read_csv(COHORT_PATH, dtype={'FOLIO_INT': 'string'})
filas = [
    {'variable_pida': 'id_persona', 'en_cohorte': 'FOLIO_INT (folio ENSANUT; no entra al estado)', 'fuente': 'Adultos', 'cobertura': 1.0},
    {'variable_pida': 'edad', 'en_cohorte': 'edad', 'fuente': 'Adultos', 'cobertura': cohorte_completa['edad'].notna().mean()},
    {'variable_pida': 'sexo', 'en_cohorte': 'sexo', 'fuente': 'Adultos', 'cobertura': cohorte_completa['sexo'].notna().mean()},
    {'variable_pida': 'diabetes_diagnosticada', 'en_cohorte': 'criterio de inclusión a0301 = 1', 'fuente': 'Adultos', 'cobertura': 1.0},
    {'variable_pida': 'comorbilidades', 'en_cohorte': 'num_complicaciones, complicacion_*, depresion', 'fuente': 'Adultos', 'cobertura': 1.0},
]
archivos_crudos = [RAW_DATA_DIR / f for f in ('Adultos_ENSANUT_2022.sav', 'Antro_Tension_ENSANUT_2022.sav', 'Muestras_Sangre_ENSANUT_2022.sav')]
try:
    faltan_crudos = [p.name for p in archivos_crudos if not p.is_file()]
    if faltan_crudos:
        raise FileNotFoundError(', '.join(faltan_crudos))
    import pyreadstat
    adultos, _ = pyreadstat.read_sav(str(RAW_DATA_DIR / 'Adultos_ENSANUT_2022.sav'), usecols=['FOLIO_INT', 'a0401'] + ([] if 'entidad' in cohorte_completa else ['entidad']))
    adultos['FOLIO_INT'] = adultos['FOLIO_INT'].astype(str).str.strip()
    m = cohorte_completa.merge(adultos, on='FOLIO_INT', how='left')
    filas.append({'variable_pida': 'entidad_federativa', 'en_cohorte': 'entidad (V2.16, solo equidad)' if 'entidad' in cohorte_completa else 'no incluida', 'fuente': 'Adultos: entidad', 'cobertura': m['entidad'].notna().mean() if 'entidad' in m else np.nan})
    filas.append({'variable_pida': 'comorbilidades (hipertensión)', 'en_cohorte': 'no incluida', 'fuente': 'Adultos: a0401', 'cobertura': m['a0401'].isin([1, 3]).mean()})
    antro, _ = pyreadstat.read_sav(str(RAW_DATA_DIR / 'Antro_Tension_ENSANUT_2022.sav'), usecols=['FOLIO_INT', 'an01_1', 'an04_1'])
    antro['FOLIO_INT'] = antro['FOLIO_INT'].astype(str).str.strip()
    m = cohorte_completa.merge(antro, on='FOLIO_INT', how='left')
    valido = m['an01_1'].between(25, 250) & m['an04_1'].between(100, 220)
    filas.append({'variable_pida': 'imc', 'en_cohorte': 'no incluida', 'fuente': 'Antro_Tension: an01_1 / an04_1 (submuestra)', 'cobertura': valido.mean()})
    _, meta = pyreadstat.read_sav(str(RAW_DATA_DIR / 'Muestras_Sangre_ENSANUT_2022.sav'), metadataonly=True)
    tiene_glucosa = any('gluc' in (v or '').lower() for v in meta.column_names_to_labels.values())
    filas.append({'variable_pida': 'glucosa', 'en_cohorte': 'no incluida', 'fuente': 'Muestras_Sangre: ' + ('contiene glucosa' if tiene_glucosa else 'sin variable de glucosa (solo serología)'), 'cobertura': np.nan})
except FileNotFoundError as error:
    print('No se encontraron los archivos crudos para la verificación:', error)
diccionario = pd.DataFrame(filas)
display(diccionario.round(3))

## 12. Sensibilidad al descuento

Cada paso del entorno es una decisión sobre un perfil: 1 542 decisiones por mes y 18 504 por episodio. Hasta V2.15 γ se aplicaba en cada decisión. Con γ = 0.95, una recompensa que llega 100 decisiones después vale 0.95¹⁰⁰ ≈ 0.006, así que el agente casi no veía el costo de gastar capacidad en los primeros perfiles del mes.

La Definición del PIDA define el retorno como Σ γ^t r_t con t = mes. Desde V2.16 los agentes oficiales (notebooks 06–08) usan ese descuento mensual. Esta sección compara, con Q-learning y el mismo presupuesto de entrenamiento:

1. Descuento mensual con γ = 0.95 (configuración oficial).
2. Descuento por decisión con γ = 0.95 (configuración V2.15) y γ = 0.9995.
3. La regla diagnóstica «alto → acción 1, resto → 0», que mide el margen de mejora del problema. No forma parte del conjunto de referencias.

Todo se evalúa con 30 episodios comunes en el escenario base. Los agentes se guardan en `sensibilidad_gamma/` y se reutilizan si ya existen.

In [ ]:
EJECUTAR_SENSIBILIDAD_GAMMA = True
VARIANTES_DESCUENTO = (('mensual', 0.95), ('decision', 0.95), ('decision', 0.9995))
EPISODIOS_GAMMA = 3 if MODO_PRUEBA else 400
SEMILLAS_GAMMA = tuple(range(5000, 5004)) if MODO_PRUEBA else tuple(range(5000, 5030))
from src.environment.diabetes_followup_env import DiabetesFollowUpEnv
from src.policies.q_learning import entrenar_q_learning

def regla_diagnostica_alto_1(env):
    return 1 if env.perfil(env.idx_actual)['categoria_riesgo'] == 'alto' else 0

sens_dir = OUTPUT_DIR / 'sensibilidad_gamma'
sens_dir.mkdir(parents=True, exist_ok=True)
sensibilidad_gamma = pd.DataFrame()
if EJECUTAR_SENSIBILIDAD_GAMMA:
    pol_gamma, huellas_gamma = {'Reglas simples': politicas['Reglas simples'],
                                'Diagnóstica: alto→1': regla_diagnostica_alto_1}, {'Reglas simples': 'regla', 'Diagnóstica: alto→1': 'regla'}
    for modo, g in VARIANTES_DESCUENTO:
        ruta = sens_dir / f'q_learning_{modo}_gamma_{g}_{EPISODIOS_GAMMA}ep_v216.npz'
        if ruta.is_file():
            agente_g = QLearningAgent.cargar(ruta, seed=20_000)
        else:
            t0 = time.time()
            agente_g, _ = entrenar_q_learning(
                lambda: DiabetesFollowUpEnv(cohorte, CAPACIDAD_MENSUAL, COSTOS_ACCION, HORIZONTE_MESES),
                episodios=EPISODIOS_GAMMA, semilla=10_000, gamma=g, epsilon_decay=0.99, descuento=modo)
            agente_g.guardar(ruta)
            print(f'{modo}, γ = {g}: entrenado en {time.time() - t0:.0f} s')
        nombre = f'Q-learning {modo} γ={g}'
        pol_gamma[nombre] = crear_politica_q_learning(agente_g)
        huellas_gamma[nombre] = hashlib.sha256(ruta.read_bytes()).hexdigest()[:16]
    esc_base = pida.Escenario('base', 'base', CAPACIDAD_MENSUAL, COSTOS_ACCION)
    ep_g = pd.concat([pida.evaluar_con_cache(esc_base, cohorte, n, p, SEMILLAS_GAMMA, CACHE_DIR / 'gamma', huellas_gamma[n],
                                             HORIZONTE_MESES, GAMMA)[0] for n, p in pol_gamma.items()], ignore_index=True)
    assert ep_g['violaciones_capacidad'].sum() == 0
    res_g = pida.resumen_con_ic(ep_g)
    sensibilidad_gamma = res_g[res_g['metrica'].isin(['recompensa_acumulada', 'cobertura_alto_riesgo'])].pivot_table(
        index='algoritmo', columns='metrica', values=['media', 'ic95_inf', 'ic95_sup'], sort=False)
    sensibilidad_gamma.columns = [f'{m}_{e}' for e, m in sensibilidad_gamma.columns]
    ref_g = sensibilidad_gamma.loc['Reglas simples', 'recompensa_acumulada_media']
    sensibilidad_gamma['mejora_vs_reglas_simples'] = (sensibilidad_gamma['recompensa_acumulada_media'] - ref_g) / abs(ref_g)
    sensibilidad_gamma = sensibilidad_gamma.reset_index()
    display(sensibilidad_gamma.round(4))

## 13. Exportación

In [ ]:
import json

salidas = {
    'criterios_pida.csv': criterios,
    'rendimiento_ic.csv': resumen[resumen['escenario'] == 'base'],
    'robustez_resumen_ic.csv': resumen,
    'robustez_escenarios.csv': robustez,
    'brechas_subgrupos_pida.csv': brechas,
    'episodios_pida.csv': episodios,
    'verificacion_diccionario.csv': diccionario,
    'sensibilidad_gamma.csv': sensibilidad_gamma,
}
for nombre, tabla in salidas.items():
    tabla.to_csv(OUTPUT_DIR / nombre, index=False)
manifiesto = {
    'commit': COMMIT, 'modo_prueba': MODO_PRUEBA, 'n_perfiles': int(len(cohorte)),
    'capacidad_mensual': int(CAPACIDAD_MENSUAL), 'costos_accion': COSTOS_ACCION, 'horizonte': HORIZONTE_MESES,
    'gamma': GAMMA, 'descuento_agentes': 'mensual', 'orden_aleatorio': True, 'version_entorno': DiabetesFollowUpEnv.VERSION, 'semillas_base': list(SEMILLAS_BASE), 'semillas_robustez': list(SEMILLAS_ROBUSTEZ),
    'escenarios': [e.__dict__ for e in ESCENARIOS], 'huellas_politica': HUELLAS_POLITICA,
    'generado_utc': datetime.now(timezone.utc).isoformat(),
}
(OUTPUT_DIR / 'manifiesto_pida.json').write_text(json.dumps(manifiesto, ensure_ascii=False, indent=2, default=str), encoding='utf-8')
print('Archivos exportados en', OUTPUT_DIR)
for p in sorted(OUTPUT_DIR.glob('*.*')):
    print('-', p.name, p.stat().st_size, 'bytes')

## Interpretación

- Los criterios con estado «Reportado» requieren evidencia externa a este notebook, como la ejecución desde cero en Colab.
- La meta aspiracional (≥ 10 % sobre la mejor regla) y la meta de robustez (≥ 80 % de escenarios) son metas, no condiciones de validez. Si no se alcanzan, el resultado se documenta junto con este análisis de sensibilidad.
- Con capacidad 100 y 1 542 perfiles hay 0.065 unidades por persona al mes. La utilización es del 100 % en todas las políticas. Desde V2.16 el orden de atención se baraja cada mes, de modo que ninguna entidad queda siempre al final de la fila.
- La sección 12 separa dos preguntas: si el problema tiene margen de mejora (regla diagnóstica) y cuánto cambia el agente con el descuento mensual frente al descuento por decisión. Las metas no se ajustan después de ver los resultados.